### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="seismic_bumps",
    dataset_year="2013",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5W902",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/seismic_bumps/ && wget -P local-data-warehouse/seismic_bumps/ https://archive.ics.uci.edu/static/public/266/seismic+bumps.zip && unzip local-data-warehouse/seismic_bumps/seismic+bumps.zip -d local-data-warehouse/seismic_bumps/
""",
    # References
    academic_reference_bibtex=r"""@article{sikora2010application,
  title={Application of rule induction algorithms for analysis of data collected by seismic hazard monitoring systems in coal mines},
  author={Sikora, Marek and Wr{\'o}bel, {\L}ukasz},
  journal={Archives of Mining Sciences},
  volume={55},
  number={1},
  pages={91--114},
  year={2010}
}
""",
    academic_reference_bibtex_key="sikora2010application",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature and reversed the values' ordinal encoding.
- We drop the constant columns "nbumps6", "nbumps7", and "nbumps89".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="HighEnergySeismicBump",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="HighEnergySeismicBump",
)

## Preprocessing

In [2]:
import arff
import pandas as pd

with open(f"{dataset_mold.path}/seismic-bumps.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])
target_feature = "HighEnergySeismicBump"
df = df.rename(columns={"class": target_feature})
df[target_feature] = df[target_feature].map({"1": "Yes", "0": "No"})

# Drop constant columns
df = df.loc[:, (df != df.iloc[0]).any()]

cat_features = [
    "seismic",
    "seismoacoustic",
    "shift",
    "ghazard",
    "HighEnergySeismicBump",
]

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,584
Columns: 16
Use sampling: False (sample size: 2,584)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['genergy', 'gpuls', 'gdenergy', 'gdpuls', 'energy', 'maxenergy', 'nbumps', 'nbumps3', 'nbumps2', 'nbumps4']
Rows remaining as candidates after top-10 filter: 12 (of 2,584)

#### Duplicate Report
Total duplicate rows: 6 (0.23% of dataset)
Duplicate rows ignoring target: 6 (0.23% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,seismic,seismoacoustic,shift,genergy,gpuls,gdenergy,gdpuls,ghazard,nbumps,nbumps2,nbumps3,nbumps4,nbumps5,energy,maxenergy,HighEnergySeismicBump
0,a,a,N,5280.0,178.0,-29.0,-16.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No
1,b,a,W,38170.0,784.0,-27.0,12.0,a,3.0,1.0,2.0,0.0,0.0,3500.0,2000.0,No
2,b,a,N,10020.0,370.0,0.0,6.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No
3,b,a,W,292040.0,1233.0,-33.0,-29.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No
4,a,a,W,16130.0,322.0,2.0,2.0,a,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,seismic,category,0.0,0.0,2.0,"a, b"
1,seismoacoustic,category,0.0,0.0,3.0,"a, b, c"
2,shift,category,0.0,0.0,2.0,"W, N"
3,ghazard,category,0.0,0.0,3.0,"a, b, c"
4,HighEnergySeismicBump,category,0.0,0.0,2.0,"No, Yes"
5,genergy,float64,0.0,0.0,2212.0,"7400.0, 6790.0, 20160.0, 15300.0, 11230.0, 19420.0, 3610.0, 19120.0, 21900.0, 8690.0"
6,gpuls,float64,0.0,0.0,1128.0,"19.0, 213.0, 17.0, 46.0, 53.0, 262.0, 133.0, 24.0, 527.0, 25.0"
7,gdenergy,float64,0.0,0.0,334.0,"-14.0, -32.0, -7.0, -38.0, -42.0, -31.0, -10.0, -40.0, -20.0, -16.0"
8,gdpuls,float64,0.0,0.0,292.0,"-32.0, 0.0, -14.0, 2.0, -28.0, 6.0, -42.0, -40.0, -6.0, -2.0"
9,nbumps,float64,0.0,0.0,10.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 7.0, 9.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
genergy,2584.0,90242.523220,229200.508894,100.0,2595650.0
gpuls,2584.0,538.579334,562.652536,2.0,4518.0
gdenergy,2584.0,12.375774,80.319051,-96.0,1245.0
gdpuls,2584.0,4.508901,63.166556,-96.0,838.0
nbumps,2584.0,0.859520,1.364616,0.0,9.0
nbumps2,2584.0,0.393576,0.783772,0.0,8.0
nbumps3,2584.0,0.392802,0.769710,0.0,7.0
nbumps4,2584.0,0.067724,0.279059,0.0,3.0
nbumps5,2584.0,0.004644,0.068001,0.0,1.0
energy,2584.0,4975.270898,20450.833222,0.0,402000.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                rank                    
HighEnergySeismicBump 1       No   2414  93.42
                      2      Yes    170   6.58
ghazard               1        a   2342  90.63
                      2        b    212   8.20
                      3        c     30   1.16
seismic               1        a   1682  65.09
                      2        b    902  34.91
seismoacoustic        1        a   1580  61.15
                      2        b    956  37.00
                      3        c     48   1.86
shift                 1        W   1663  64.36
                      2        N    921  35.64

In [8]:
# Target Distribution
target_df

,count,pct
HighEnergySeismicBump,,
No,2414,93.42
Yes,170,6.58


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to seismic_bumps/019d7369-a6b4-7060-a145-dc996b59cf1b


019d7369-a6b4-7060-a145-dc996b59cf1b
df6b22ee162c22d06dab190d119400c34b11f67d28c83530663f9847ae1a0b5d
